In [16]:
# ============================================================
# Structured Data Analysis Using NumPy: IPL Case Study
# Rules Followed:
# ✔ No Pandas
# ✔ Only NumPy + Core Python
# ✔ Vectorized operations preferred
# ============================================================

import numpy as np

# ============================================================
# Task 0: Data Loading and Preprocessing
# ============================================================

# Load deliveries.csv
deliveries = np.genfromtxt(
    "deliveries.csv",
    delimiter=",",
    skip_header=1,
    dtype=None,
    encoding="utf-8",
    missing_values="",
    filling_values=0
)

# Load matches.csv
matches = np.genfromtxt(
    "matches.csv",
    delimiter=",",
    skip_header=1,
    dtype=str,
    encoding="utf-8",
    autostrip=True,
    quotechar='"'
)

# -------------------------
# Extract columns deliveries
# -------------------------
match_ids      = deliveries[:,0].astype(int)
inning         = deliveries[:,1].astype(int)
batting_team   = deliveries[:,2]
bowling_team   = deliveries[:,3]
over           = deliveries[:,4].astype(int)
ball           = deliveries[:,5].astype(int)
batter         = deliveries[:,6]
bowler         = deliveries[:,7]
batsman_runs   = deliveries[:,9].astype(int)
extra_runs     = deliveries[:,10].astype(int)
total_runs     = deliveries[:,11].astype(int)

# -------------------------
# Extract columns matches
# -------------------------
match_id_m     = matches[:,0].astype(int)
team1          = matches[:,7]
team2          = matches[:,8]
toss_winner    = matches[:,9]
winner         = matches[:,11]

print("Data Loaded Successfully")


# ============================================================
# Task 1: Total Runs per Match
# ============================================================

unique_matches = np.unique(match_ids)

runs_per_match = np.array([
    (mid, batsman_runs[match_ids == mid].sum())
    for mid in unique_matches
], dtype=object)

print("\nTask 1: Total Runs Per Match")
print(runs_per_match[:10])


# ============================================================
# Task 2: Top 5 Batters
# ============================================================

unique_batters = np.unique(batter)

batter_runs = np.array([
    batsman_runs[batter == b].sum()
    for b in unique_batters
])

top5_idx = np.argsort(batter_runs)[-5:][::-1]

top5_batters = np.array([
    (unique_batters[i], batter_runs[i])
    for i in top5_idx
], dtype=object)

print("\nTask 2: Top 5 Batters")
print(top5_batters)


# ============================================================
# Task 3: Strike Rate Calculation
# ============================================================

balls_faced = np.array([
    np.sum(batter == b)
    for b in unique_batters
])

strike_rate = (batter_runs / balls_faced) * 100

print("\nTask 3: Strike Rate (Top 10)")
for i in range(10):
    print(unique_batters[i], round(strike_rate[i],2))


# ============================================================
# Task 4: Economy Rate of Bowlers
# ============================================================

unique_bowlers = np.unique(bowler)

runs_given = np.array([
    total_runs[bowler == bw].sum()
    for bw in unique_bowlers
])

balls_bowled = np.array([
    np.sum(bowler == bw)
    for bw in unique_bowlers
])

overs_bowled = balls_bowled / 6

economy = runs_given / overs_bowled

print("\nTask 4: Economy Rate (Top 10)")
for i in range(10):
    print(unique_bowlers[i], round(economy[i],2))


# ============================================================
# Task 5: Runs per Over (1 to 20)
# ============================================================

avg_runs_over = np.array([
    batsman_runs[over == ov].mean()
    for ov in range(1, 21)
])

print("\nTask 5: Average Runs Per Over")
print(avg_runs_over)


# ============================================================
# Task 6: Boundary Analysis
# ============================================================

fours = np.sum(batsman_runs == 4)
sixes = np.sum(batsman_runs == 6)

print("\nTask 6:")
print("Total Fours :", fours)
print("Total Sixes :", sixes)

# Team with most boundaries
boundary_mask = (batsman_runs == 4) | (batsman_runs == 6)

teams = np.unique(batting_team)

team_boundaries = np.array([
    np.sum(boundary_mask & (batting_team == t))
    for t in teams
])

max_team = teams[np.argmax(team_boundaries)]

print("Most Boundaries Team:", max_team)


# ============================================================
# Task 7: Death Overs Analysis (16–20)
# ============================================================

death_mask = (over >= 16) & (over <= 20)

death_total = batsman_runs[death_mask].sum()

death_team_runs = np.array([
    batsman_runs[death_mask & (batting_team == t)].sum()
    for t in teams
])

best_death_team = teams[np.argmax(death_team_runs)]

print("\nTask 7:")
print("Total Death Overs Runs:", death_total)
print("Highest Scoring Team in Death Overs:", best_death_team)


# ============================================================
# Task 8: Highest Scoring Match
# ============================================================

highest_idx = np.argmax(runs_per_match[:,1].astype(int))

print("\nTask 8:")
print("Highest Scoring Match:", runs_per_match[highest_idx])


# ============================================================
# Task 9: Match Winner Approximation
# ============================================================

print("\nTask 9: Match Winner Approximation")

for mid in unique_matches[:10]:

    teams_in_match = np.unique(batting_team[match_ids == mid])

    if len(teams_in_match) == 2:
        t1, t2 = teams_in_match

        r1 = batsman_runs[(match_ids == mid) & (batting_team == t1)].sum()
        r2 = batsman_runs[(match_ids == mid) & (batting_team == t2)].sum()

        winner = t1 if r1 > r2 else t2

        print(mid, "Winner:", winner)


# ============================================================
# Task 10: Toss Impact Analysis
# ============================================================

print("\nTask 10: Toss Impact")

for mid in match_id_m[:10]:

    toss = toss_winner[match_id_m == mid][0]

    mask = match_ids == mid

    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        opp = teams_played[teams_played != toss][0]

        toss_runs = batsman_runs[mask & (batting_team == toss)].sum()
        opp_runs  = batsman_runs[mask & (batting_team == opp)].sum()

        result = toss_runs > opp_runs

        print(mid, toss, "Scored More?", result)


# ============================================================
# Task 11: Match Scorecard Generation
# ============================================================

print("\nTask 11: Match Scorecards")

for mid in unique_matches[:5]:

    mask = match_ids == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        t1, t2 = teams_played

        r1 = batsman_runs[mask & (batting_team == t1)].sum()
        r2 = batsman_runs[mask & (batting_team == t2)].sum()

        print("\nMatch:", mid)
        print(t1, ":", r1)
        print(t2, ":", r2)

# ============================================================
# END
# ============================================================

TypeError: genfromtxt() got an unexpected keyword argument 'quotechar'

In [17]:
# ===============================================================
# IPL DATA ANALYSIS PROJECT
# NumPy + Core Python Only
# No Pandas
# Compatible with older NumPy versions
# ===============================================================

import numpy as np
import csv

# ===============================================================
# TASK 0 : LOAD DATA
# ===============================================================

# ---------- Load matches.csv using csv module ----------
with open("matches.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header_matches = next(reader)
    matches = np.array(list(reader), dtype=str)

# ---------- Load deliveries.csv using NumPy ----------
deliveries = np.genfromtxt(
    "deliveries.csv",
    delimiter=",",
    skip_header=1,
    dtype=str,
    encoding="utf-8"
)

print("Matches Shape    :", matches.shape)
print("Deliveries Shape :", deliveries.shape)


# ===============================================================
# EXTRACT COLUMNS
# ===============================================================

# deliveries.csv
match_id      = deliveries[:,0].astype(int)
inning        = deliveries[:,1].astype(int)
batting_team  = deliveries[:,2]
bowling_team  = deliveries[:,3]
over          = deliveries[:,4].astype(int)
ball          = deliveries[:,5].astype(int)
batter        = deliveries[:,6]
bowler        = deliveries[:,7]
batsman_runs  = deliveries[:,9].astype(int)
extra_runs    = deliveries[:,10].astype(int)
total_runs    = deliveries[:,11].astype(int)

# matches.csv
m_id          = matches[:,0].astype(int)
team1         = matches[:,7]
team2         = matches[:,8]
toss_winner   = matches[:,9]
winner        = matches[:,11]


# ===============================================================
# TASK 1 : TOTAL RUNS PER MATCH
# ===============================================================

unique_matches = np.unique(match_id)

runs_per_match = np.array([
    (mid, batsman_runs[match_id == mid].sum())
    for mid in unique_matches
], dtype=object)

print("\nTASK 1 : Total Runs Per Match")
print(runs_per_match[:10])


# ===============================================================
# TASK 2 : TOP 5 BATTERS
# ===============================================================

unique_batters = np.unique(batter)

runs_batter = np.array([
    batsman_runs[batter == p].sum()
    for p in unique_batters
])

top5_idx = np.argsort(runs_batter)[-5:][::-1]

print("\nTASK 2 : Top 5 Batters")

for i in top5_idx:
    print(unique_batters[i], ":", runs_batter[i])


# ===============================================================
# TASK 3 : STRIKE RATE
# ===============================================================

balls_faced = np.array([
    np.sum(batter == p)
    for p in unique_batters
])

strike_rate = (runs_batter / balls_faced) * 100

print("\nTASK 3 : Strike Rate (Top 10)")
for i in range(10):
    print(unique_batters[i], ":", round(strike_rate[i],2))


# ===============================================================
# TASK 4 : ECONOMY RATE
# ===============================================================

unique_bowlers = np.unique(bowler)

runs_given = np.array([
    total_runs[bowler == bw].sum()
    for bw in unique_bowlers
])

balls_bowled = np.array([
    np.sum(bowler == bw)
    for bw in unique_bowlers
])

overs_bowled = balls_bowled / 6

economy = runs_given / overs_bowled

print("\nTASK 4 : Economy Rate (Top 10)")
for i in range(10):
    print(unique_bowlers[i], ":", round(economy[i],2))


# ===============================================================
# TASK 5 : AVG RUNS PER OVER
# ===============================================================

avg_runs = np.array([
    batsman_runs[over == ov].mean()
    for ov in range(1,21)
])

print("\nTASK 5 : Average Runs Per Over")
print(avg_runs)


# ===============================================================
# TASK 6 : BOUNDARIES
# ===============================================================

fours = np.sum(batsman_runs == 4)
sixes = np.sum(batsman_runs == 6)

print("\nTASK 6 : Boundary Analysis")
print("Total Fours :", fours)
print("Total Sixes :", sixes)

boundary_mask = (batsman_runs == 4) | (batsman_runs == 6)

teams = np.unique(batting_team)

team_boundaries = np.array([
    np.sum(boundary_mask & (batting_team == t))
    for t in teams
])

best_team = teams[np.argmax(team_boundaries)]

print("Most Boundaries By :", best_team)


# ===============================================================
# TASK 7 : DEATH OVERS ANALYSIS
# ===============================================================

death_mask = (over >= 16) & (over <= 20)

death_runs = batsman_runs[death_mask].sum()

team_death = np.array([
    batsman_runs[death_mask & (batting_team == t)].sum()
    for t in teams
])

best_death_team = teams[np.argmax(team_death)]

print("\nTASK 7 : Death Overs")
print("Total Runs :", death_runs)
print("Highest Team :", best_death_team)


# ===============================================================
# TASK 8 : HIGHEST SCORING MATCH
# ===============================================================

idx = np.argmax(runs_per_match[:,1].astype(int))

print("\nTASK 8 : Highest Scoring Match")
print(runs_per_match[idx])


# ===============================================================
# TASK 9 : MATCH WINNER APPROXIMATION
# ===============================================================

print("\nTASK 9 : Winner Approximation")

for mid in unique_matches[:10]:

    mask = match_id == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        t1 = teams_played[0]
        t2 = teams_played[1]

        r1 = batsman_runs[mask & (batting_team == t1)].sum()
        r2 = batsman_runs[mask & (batting_team == t2)].sum()

        predicted = t1 if r1 > r2 else t2

        print(mid, ":", predicted)


# ===============================================================
# TASK 10 : TOSS IMPACT
# ===============================================================

print("\nTASK 10 : Toss Impact")

for mid in m_id[:10]:

    toss = toss_winner[m_id == mid][0]

    mask = match_id == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        other = teams_played[teams_played != toss][0]

        toss_runs = batsman_runs[mask & (batting_team == toss)].sum()
        opp_runs  = batsman_runs[mask & (batting_team == other)].sum()

        print(mid, ":", toss, "Scored More?", toss_runs > opp_runs)


# ===============================================================
# TASK 11 : SCORECARD
# ===============================================================

print("\nTASK 11 : Match Scorecards")

for mid in unique_matches[:5]:

    mask = match_id == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        t1 = teams_played[0]
        t2 = teams_played[1]

        r1 = batsman_runs[mask & (batting_team == t1)].sum()
        r2 = batsman_runs[mask & (batting_team == t2)].sum()

        print("\nMatch :", mid)
        print(t1, ":", r1)
        print(t2, ":", r2)

# ===============================================================
# END
# ===============================================================

Matches Shape    : (1095, 20)
Deliveries Shape : (260920, 17)

TASK 1 : Total Runs Per Match
[[np.int64(335982) np.int64(268)]
 [np.int64(335983) np.int64(430)]
 [np.int64(335984) np.int64(244)]
 [np.int64(335985) np.int64(315)]
 [np.int64(335986) np.int64(184)]
 [np.int64(335987) np.int64(318)]
 [np.int64(335988) np.int64(268)]
 [np.int64(335989) np.int64(379)]
 [np.int64(335990) np.int64(418)]
 [np.int64(335991) np.int64(284)]]

TASK 2 : Top 5 Batters
V Kohli : 8014
S Dhawan : 6769
RG Sharma : 6630
DA Warner : 6567
SK Raina : 5536

TASK 3 : Strike Rate (Top 10)
A Ashish Reddy : 142.86
A Badoni : 125.54
A Chandila : 57.14
A Chopra : 70.67
A Choudhary : 125.0
A Dananjaya : 80.0
A Flintoff : 108.77
A Kamboj : 100.0
A Kumble : 71.43
A Manohar : 127.62

TASK 4 : Economy Rate (Top 10)
A Ashish Reddy : 8.89
A Badoni : 8.88
A Chandila : 6.28
A Choudhary : 8.0
A Dananjaya : 11.28
A Flintoff : 9.64
A Kamboj : 10.15
A Kumble : 6.65
A Mishra : 7.3
A Mithun : 9.17

TASK 5 : Average Runs Per Over


C:\Users\rkas3010\AppData\Local\Temp\ipykernel_43936\1876292077.py:139: RuntimeWarning: Mean of empty slice
  batsman_runs[over == ov].mean()
c:\Users\rkas3010\OneDrive - 7-Eleven, Inc\Desktop\bootcamp\Development\Python\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


Most Boundaries By : Mumbai Indians

TASK 7 : Death Overs
Total Runs : 71303
Highest Team : Mumbai Indians

TASK 8 : Highest Scoring Match
[np.int64(1426268) np.int64(520)]

TASK 9 : Winner Approximation
335982 : Kolkata Knight Riders
335983 : Chennai Super Kings
335984 : Rajasthan Royals
335985 : Royal Challengers Bangalore
335986 : Deccan Chargers
335987 : Kings XI Punjab
335988 : Deccan Chargers
335989 : Chennai Super Kings
335990 : Rajasthan Royals
335991 : Kings XI Punjab

TASK 10 : Toss Impact
335982 : Royal Challengers Bangalore Scored More? False
335983 : Chennai Super Kings Scored More? True
335984 : Rajasthan Royals Scored More? False
335985 : Mumbai Indians Scored More? False
335986 : Deccan Chargers Scored More? True
335987 : Kings XI Punjab Scored More? True
335988 : Deccan Chargers Scored More? True
335989 : Mumbai Indians Scored More? False
335990 : Rajasthan Royals Scored More? True
335991 : Mumbai Indians Scored More? False

TASK 11 : Match Scorecards

Match : 335982
K

In [1]:
# ===============================================================
# IPL DATA ANALYSIS PROJECT (CORRECTED VERSION)
# NumPy + Core Python Only
# No Pandas
# Compatible with older NumPy versions
# ===============================================================

import numpy as np
import csv

# ===============================================================
# TASK 0 : LOAD DATA
# ===============================================================

# ---------- Load matches.csv ----------
with open("matches.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    header_matches = next(reader)
    matches = np.array(list(reader), dtype=str)

# ---------- Load deliveries.csv ----------
deliveries = np.genfromtxt(
    "deliveries.csv",
    delimiter=",",
    skip_header=1,
    dtype=str,
    encoding="utf-8"
)

print("Matches Shape    :", matches.shape)
print("Deliveries Shape :", deliveries.shape)


# ===============================================================
# EXTRACT COLUMNS
# ===============================================================

# deliveries.csv
match_id      = deliveries[:, 0].astype(int)
inning        = deliveries[:, 1].astype(int)
batting_team  = deliveries[:, 2]
bowling_team  = deliveries[:, 3]
over          = deliveries[:, 4].astype(int)      # 0 to 19
ball          = deliveries[:, 5].astype(int)
batter        = deliveries[:, 6]
bowler        = deliveries[:, 7]
batsman_runs  = deliveries[:, 9].astype(int)
extra_runs    = deliveries[:,10].astype(int)
total_runs    = deliveries[:,11].astype(int)

# matches.csv
m_id          = matches[:,0].astype(int)
team1         = matches[:,7]
team2         = matches[:,8]
toss_winner   = matches[:,9]
winner        = matches[:,11]


# ===============================================================
# TASK 1 : TOTAL RUNS PER MATCH
# ===============================================================

unique_matches = np.unique(match_id)

runs_per_match = np.array([
    (mid, total_runs[match_id == mid].sum())
    for mid in unique_matches
], dtype=object)

print("\nTASK 1 : Total Runs Per Match")
print(runs_per_match[:10])


# ===============================================================
# TASK 2 : TOP 5 BATTERS
# ===============================================================

unique_batters = np.unique(batter)

runs_batter = np.array([
    batsman_runs[batter == p].sum()
    for p in unique_batters
])

top5_idx = np.argsort(runs_batter)[-5:][::-1]

print("\nTASK 2 : Top 5 Batters")
for i in top5_idx:
    print(unique_batters[i], ":", runs_batter[i])


# ===============================================================
# TASK 3 : STRIKE RATE
# ===============================================================

balls_faced = np.array([
    np.sum(batter == p)
    for p in unique_batters
])

strike_rate = (runs_batter / balls_faced) * 100

print("\nTASK 3 : Strike Rate (Top 10)")
for i in range(min(10, len(unique_batters))):
    print(unique_batters[i], ":", round(strike_rate[i], 2))


# ===============================================================
# TASK 4 : ECONOMY RATE
# ===============================================================

unique_bowlers = np.unique(bowler)

runs_given = np.array([
    total_runs[bowler == bw].sum()
    for bw in unique_bowlers
])

balls_bowled = np.array([
    np.sum(bowler == bw)
    for bw in unique_bowlers
])

overs_bowled = balls_bowled / 6
economy = runs_given / overs_bowled

print("\nTASK 4 : Economy Rate (Top 10)")
for i in range(min(10, len(unique_bowlers))):
    print(unique_bowlers[i], ":", round(economy[i], 2))


# ===============================================================
# TASK 5 : AVG RUNS PER OVER (OVERS 1 TO 20)
# Dataset stores over = 0 to 19
# ===============================================================

# avg_runs = np.array([
#     total_runs[over == ov].mean()
#     for ov in range(0, 20)
# ])

# print("\nTASK 5 : Average Runs Per Over")
# for i, val in enumerate(avg_runs, start=1):
#     print("Over", i, ":", round(val, 2))

avg_runs = []
for ov in range(0, 20):
    over_totals = []
    for mid in unique_matches:
        for inn in [1, 2]:
            mask = (match_id == mid) & (inning == inn) & (over == ov)
            if np.any(mask):
                over_runs = total_runs[mask].sum()
                over_totals.append(over_runs)
    if len(over_totals) > 0:
        avg_runs.append(np.mean(over_totals))
    else:
        avg_runs.append(0)
avg_runs = np.array(avg_runs)
print("\nTASK 5 : Average Runs Per Over (Corrected)")
for i, val in enumerate(avg_runs, start=1):
    print("Over", i, ":", round(val, 2))


# ===============================================================
# TASK 6 : BOUNDARY ANALYSIS
# ===============================================================

fours = np.sum(batsman_runs == 4)
sixes = np.sum(batsman_runs == 6)

print("\nTASK 6 : Boundary Analysis")
print("Total Fours :", fours)
print("Total Sixes :", sixes)

boundary_mask = (batsman_runs == 4) | (batsman_runs == 6)

teams = np.unique(batting_team)

team_boundaries = np.array([
    np.sum(boundary_mask & (batting_team == t))
    for t in teams
])

best_team = teams[np.argmax(team_boundaries)]

print("Most Boundaries By :", best_team)


# ===============================================================
# TASK 7 : DEATH OVERS ANALYSIS
# Dataset overs 16-20 => stored as 15 to 19
# ===============================================================

death_mask = (over >= 15) & (over <= 19)

death_runs = total_runs[death_mask].sum()

team_death = np.array([
    total_runs[death_mask & (batting_team == t)].sum()
    for t in teams
])

best_death_team = teams[np.argmax(team_death)]

print("\nTASK 7 : Death Overs")
print("Total Runs :", death_runs)
print("Highest Team :", best_death_team)


# ===============================================================
# TASK 8 : HIGHEST SCORING MATCH
# ===============================================================

idx = np.argmax(runs_per_match[:,1].astype(int))

print("\nTASK 8 : Highest Scoring Match")
print(runs_per_match[idx])


# ===============================================================
# TASK 9 : MATCH WINNER APPROXIMATION
# ===============================================================

print("\nTASK 9 : Winner Approximation")

for mid in unique_matches[:10]:

    mask = match_id == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        t1, t2 = teams_played

        r1 = total_runs[mask & (batting_team == t1)].sum()
        r2 = total_runs[mask & (batting_team == t2)].sum()

        predicted = t1 if r1 > r2 else t2

        print(mid, ":", predicted)


# ===============================================================
# TASK 10 : TOSS IMPACT
# ===============================================================

print("\nTASK 10 : Toss Impact")

for mid in m_id[:10]:

    toss = toss_winner[m_id == mid][0]

    mask = match_id == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        other = teams_played[teams_played != toss][0]

        toss_runs = total_runs[mask & (batting_team == toss)].sum()
        opp_runs  = total_runs[mask & (batting_team == other)].sum()

        print(mid, ":", toss, "Scored More?", toss_runs > opp_runs)


# ===============================================================
# TASK 11 : SCORECARD
# ===============================================================

print("\nTASK 11 : Match Scorecards")

for mid in unique_matches[:5]:

    mask = match_id == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        t1, t2 = teams_played

        r1 = total_runs[mask & (batting_team == t1)].sum()
        r2 = total_runs[mask & (batting_team == t2)].sum()

        print("\nMatch :", mid)
        print(t1, ":", r1)
        print(t2, ":", r2)

# ===============================================================
# END
# ===============================================================

Matches Shape    : (1095, 20)
Deliveries Shape : (260920, 17)

TASK 1 : Total Runs Per Match
[[np.int64(335982) np.int64(304)]
 [np.int64(335983) np.int64(447)]
 [np.int64(335984) np.int64(261)]
 [np.int64(335985) np.int64(331)]
 [np.int64(335986) np.int64(222)]
 [np.int64(335987) np.int64(334)]
 [np.int64(335988) np.int64(285)]
 [np.int64(335989) np.int64(410)]
 [np.int64(335990) np.int64(431)]
 [np.int64(335991) np.int64(298)]]

TASK 2 : Top 5 Batters
V Kohli : 8014
S Dhawan : 6769
RG Sharma : 6630
DA Warner : 6567
SK Raina : 5536

TASK 3 : Strike Rate (Top 10)
A Ashish Reddy : 142.86
A Badoni : 125.54
A Chandila : 57.14
A Chopra : 70.67
A Choudhary : 125.0
A Dananjaya : 80.0
A Flintoff : 108.77
A Kamboj : 100.0
A Kumble : 71.43
A Manohar : 127.62

TASK 4 : Economy Rate (Top 10)
A Ashish Reddy : 8.89
A Badoni : 8.88
A Chandila : 6.28
A Choudhary : 8.0
A Dananjaya : 11.28
A Flintoff : 9.64
A Kamboj : 10.15
A Kumble : 6.65
A Mishra : 7.3
A Mithun : 9.17

TASK 5 : Average Runs Per Over 

In [2]:
# ===============================================================
# IPL DATA ANALYSIS PROJECT (ENGLISH PHRASE OUTPUT VERSION)
# NumPy + Core Python Only
# No Pandas
# ===============================================================

import numpy as np
import csv

# ===============================================================
# LOAD DATA
# ===============================================================

with open("matches.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    matches = np.array(list(reader), dtype=str)

deliveries = np.genfromtxt(
    "deliveries.csv",
    delimiter=",",
    skip_header=1,
    dtype=str,
    encoding="utf-8"
)

print("The matches dataset contains", matches.shape[0], "rows and", matches.shape[1], "columns.")
print("The deliveries dataset contains", deliveries.shape[0], "rows and", deliveries.shape[1], "columns.")


# ===============================================================
# EXTRACT COLUMNS
# ===============================================================

match_id      = deliveries[:,0].astype(int)
batting_team  = deliveries[:,2]
over          = deliveries[:,4].astype(int)
batter        = deliveries[:,6]
bowler        = deliveries[:,7]
batsman_runs  = deliveries[:,9].astype(int)
total_runs    = deliveries[:,11].astype(int)

m_id          = matches[:,0].astype(int)
toss_winner   = matches[:,9]


# ===============================================================
# TASK 1 : TOTAL RUNS PER MATCH
# ===============================================================

unique_matches = np.unique(match_id)

runs_per_match = np.array([
    total_runs[match_id == mid].sum()
    for mid in unique_matches
])

print("\nThe total runs scored in the first five matches are:")

for i in range(5):
    print("Match", unique_matches[i], "had", runs_per_match[i], "runs.")


# ===============================================================
# TASK 2 : TOP 5 BATTERS
# ===============================================================

unique_batters = np.unique(batter)

runs_batter = np.array([
    batsman_runs[batter == p].sum()
    for p in unique_batters
])

top5_idx = np.argsort(runs_batter)[-5:][::-1]

print("\nThe top five batters in IPL history are:")

for i in top5_idx:
    print(unique_batters[i], "scored", runs_batter[i], "runs.")


# ===============================================================
# TASK 3 : STRIKE RATE
# ===============================================================

balls_faced = np.array([
    np.sum(batter == p)
    for p in unique_batters
])

strike_rate = (runs_batter / balls_faced) * 100

best_sr = np.argmax(strike_rate)

print("\nThe batter with the highest strike rate is",
      unique_batters[best_sr],
      "with a strike rate of",
      round(strike_rate[best_sr], 2))


# ===============================================================
# TASK 4 : ECONOMY RATE
# ===============================================================

unique_bowlers = np.unique(bowler)

runs_given = np.array([
    total_runs[bowler == b].sum()
    for b in unique_bowlers
])

balls_bowled = np.array([
    np.sum(bowler == b)
    for b in unique_bowlers
])

economy = runs_given / (balls_bowled / 6)

best_eco = np.argmin(economy)

print("\nThe most economical bowler is",
      unique_bowlers[best_eco],
      "with an economy rate of",
      round(economy[best_eco], 2))


# ===============================================================
# TASK 5 : RUNS PER OVER
# ===============================================================

avg_runs = np.array([
    total_runs[over == ov].mean()
    for ov in range(0,20)
])

print("\nThe average runs scored in over 1 is",
      round(avg_runs[0],2),
      "and in over 20 is",
      round(avg_runs[19],2))


# ===============================================================
# TASK 6 : BOUNDARIES
# ===============================================================

fours = np.sum(batsman_runs == 4)
sixes = np.sum(batsman_runs == 6)

print("\nA total of", fours, "fours and", sixes, "sixes were hit in the dataset.")

boundary_mask = (batsman_runs == 4) | (batsman_runs == 6)

teams = np.unique(batting_team)

team_boundaries = np.array([
    np.sum(boundary_mask & (batting_team == t))
    for t in teams
])

best_team = teams[np.argmax(team_boundaries)]

print(best_team, "hit the highest number of boundaries.")


# ===============================================================
# TASK 7 : DEATH OVERS
# ===============================================================

death_mask = (over >= 15) & (over <= 19)

death_runs = total_runs[death_mask].sum()

team_death = np.array([
    total_runs[death_mask & (batting_team == t)].sum()
    for t in teams
])

best_death_team = teams[np.argmax(team_death)]

print("\nA total of", death_runs,
      "runs were scored in the death overs.")

print(best_death_team,
      "scored the most runs in death overs.")


# ===============================================================
# TASK 8 : HIGHEST SCORING MATCH
# ===============================================================

idx = np.argmax(runs_per_match)

print("\nThe highest scoring match was Match",
      unique_matches[idx],
      "with",
      runs_per_match[idx],
      "runs.")


# ===============================================================
# TASK 9 : WINNER APPROXIMATION
# ===============================================================

print("\nWinner approximation for first five matches:")

for mid in unique_matches[:5]:

    mask = match_id == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        t1, t2 = teams_played

        r1 = total_runs[mask & (batting_team == t1)].sum()
        r2 = total_runs[mask & (batting_team == t2)].sum()

        predicted = t1 if r1 > r2 else t2

        print("For Match", mid, ", the predicted winner is", predicted)


# ===============================================================
# TASK 10 : TOSS IMPACT
# ===============================================================

print("\nToss impact for first five matches:")

for mid in m_id[:5]:

    toss = toss_winner[m_id == mid][0]

    mask = match_id == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        other = teams_played[teams_played != toss][0]

        toss_runs = total_runs[mask & (batting_team == toss)].sum()
        opp_runs  = total_runs[mask & (batting_team == other)].sum()

        if toss_runs > opp_runs:
            print("In Match", mid, ", the toss winner scored more runs.")
        else:
            print("In Match", mid, ", the toss winner did not score more runs.")


# ===============================================================
# TASK 11 : SCORECARD
# ===============================================================

print("\nScorecards for first three matches:")

for mid in unique_matches[:3]:

    mask = match_id == mid
    teams_played = np.unique(batting_team[mask])

    if len(teams_played) == 2:

        t1, t2 = teams_played

        r1 = total_runs[mask & (batting_team == t1)].sum()
        r2 = total_runs[mask & (batting_team == t2)].sum()

        print("\nIn Match", mid)
        print(t1, "scored", r1, "runs.")
        print(t2, "scored", r2, "runs.")

# ===============================================================
# END
# ===============================================================

The matches dataset contains 1095 rows and 20 columns.
The deliveries dataset contains 260920 rows and 17 columns.

The total runs scored in the first five matches are:
Match 335982 had 304 runs.
Match 335983 had 447 runs.
Match 335984 had 261 runs.
Match 335985 had 331 runs.
Match 335986 had 222 runs.

The top five batters in IPL history are:
V Kohli scored 8014 runs.
S Dhawan scored 6769 runs.
RG Sharma scored 6630 runs.
DA Warner scored 6567 runs.
SK Raina scored 5536 runs.

The batter with the highest strike rate is L Wood with a strike rate of 300.0

The most economical bowler is AC Gilchrist with an economy rate of 0.0

The average runs scored in over 1 is 0.98 and in over 20 is 1.78

A total of 29850 fours and 13051 sixes were hit in the dataset.
Mumbai Indians hit the highest number of boundaries.

A total of 93884 runs were scored in the death overs.
Mumbai Indians scored the most runs in death overs.

The highest scoring match was Match 1426268 with 549 runs.

Winner approxim